# **PyMC Kalman Filter Price Target Model — GaussianRandomWalk state-space**

## Uses PyMC GaussianRandomWalk for latent price state with an observation model for noisy price targets.

### Schema-aligned with:
- MV: `pml.mv_pymc_kalman_pt`
- Catalogue: `SELECT * FROM pml.vw_pymc_feature_catalogue WHERE model_target = 'kalman_pt' ORDER BY pymc_role, feature_role, feature_alias`




In [ ]:
%%sql
SELECT * FROM pml.mv_pymc_kalman_pt mpkp WHERE observed_pt IS NOT NULL AND next_earnings >= '2026-01-01'

## 1. Notebook Setup & Imports

In [ ]:
%%sql
SELECT *
FROM pml.vw_pymc_feature_catalogue
WHERE model_target = 'kalman_pt'
ORDER BY pymc_role, feature_role, feature_alias

## 1. Notebook Setup & Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

# ArviZ 1.0 split-package imports: arviz-plots owns `style` + plotting,
# arviz-stats owns `summary` / `rhat` / `ess`. Address each submodule directly.
import arviz_plots as azp
import arviz_stats as azs
from arviz_plots import visuals as azv  # low-level primitives for custom composition

import pymc as pm
import pytensor.tensor as pt

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# --- Plotting backend + theme ------------------------------------------------
# Pin the arviz-plots backend so PlotCollection / plot_* render through
# matplotlib (the notebook's dark-theme target). `arviz-vibrant` is the bright
# arviz 1.x palette that reads well on a dark background — the old 0.x
# `arviz-darkgrid` style does not exist in arviz-plots 1.x (its silent failure
# in the previous revision is why the arviz plots were rendering un-themed).
azp.backend = 'matplotlib'
plt.style.use('dark_background')
try:
    azp.style.use('arviz-vibrant')
except (OSError, ValueError, AttributeError):
    pass
sns.set_theme(style='darkgrid', context='notebook',
              rc={
                  'figure.facecolor': '#1e1e1e',
                  'axes.facecolor': '#2a2a2a',
                  'savefig.facecolor': '#1e1e1e',
                  'axes.edgecolor': '#cccccc',
                  'axes.labelcolor': '#e6e6e6',
                  'xtick.color': '#e6e6e6',
                  'ytick.color': '#e6e6e6',
                  'text.color': '#e6e6e6',
                  'grid.color': '#555555',
              })

# arviz_plots builds its per-chain colour aesthetic by reshaping the *active*
# matplotlib colour cycle. seaborn.set_theme() installs that cycle as RGB
# tuples (e.g. (0.29, 0.44, 0.69)); arviz_plots then does
# `np.array(colours[:n_chains]).reshape((n_chains,))`, which for 4 chains turns
# 4 RGB triples into an array of size 12 and raises
# 'cannot reshape array of size 12 into shape (4,)'. Re-express the cycle as
# hex strings so plot_trace / PlotCollection work under the seaborn theme.
from cycler import cycler as _cycler
import matplotlib.colors as _mcolors
_cycle_cols = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
if _cycle_cols and not all(isinstance(_c, str) for _c in _cycle_cols):
    plt.rcParams['axes.prop_cycle'] = _cycler(
        color=[_mcolors.to_hex(_c) for _c in _cycle_cols]
    )
plt.rcParams['figure.dpi'] = 110

print(f'kalman_df         : {kalman_df.shape}')
print(f'feature_catalogue : {feature_catalogue.shape}')
if 'model_target' in feature_catalogue.columns:
    _mt = feature_catalogue['model_target']
    if not (_mt == 'kalman_pt').all():
        warnings.warn("feature_catalogue is not fully filtered to model_target='kalman_pt'.")

### 1.1 Custom visualization — Kalman price-target path (`arviz_plots` composition)

The headline view for this model is the **Kalman-smoothed price target over time**:
the latent state is a noisy walk through the observed analyst targets, and the value
of a state-space filter is that it returns a *credible band*, not just a point.

`plot_price_target_path()` composes that view with the `arviz_plots` low-level API
(`PlotCollection.grid` + `arviz_plots.visuals`) rather than a one-shot `plot_*`
helper — see the [compose-your-own-plot tutorial](https://python.arviz.org/projects/plots/en/latest/tutorials/compose_own_plot.html).
It layers, on a single time axis:

- **nested HDI bands** (94% + 50%) of the posterior latent `state`, darkening inward;
- the **posterior-median** smoothed path;
- the **observed** analyst price targets as points;
- a dashed **last-price** reference line.

It reads the `time` coordinate straight off the posterior (a `DatetimeIndex` when the
model was fit with `dates=`), so it works for both the single-ISIN time-series filter
(Section 11) and any future per-ISIN posterior carrying a `state`/`time` pair.

In [ ]:
from typing import Optional, Sequence


def plot_price_target_path(
    idata,
    *,
    state_var: str = "state",
    observed: Optional[np.ndarray] = None,
    dates: Optional[pd.DatetimeIndex] = None,
    last_price: Optional[float] = None,
    ticker: Optional[str] = None,
    hdi_probs: Sequence[float] = (0.94, 0.5),
    figsize: tuple[float, float] = (11, 5),
    color: str = "#56b4e9",
    observed_color: str = "#ffb000",
):
    """Compose a Kalman-smoothed price-target trajectory with ``arviz_plots``.

    Builds the plot from the low-level composition API
    (:meth:`arviz_plots.PlotCollection.grid` + :mod:`arviz_plots.visuals`) so the
    posterior latent state, its credible bands, and the raw observations share a
    single time axis.

    Parameters
    ----------
    idata
        Inference object whose ``posterior`` holds ``state_var`` over a ``time``
        dim (e.g. the output of :meth:`KalmanFilterPriceTarget.fit`).
    state_var
        Posterior variable carrying the price-space latent state. Default ``"state"``.
    observed
        Observed analyst price targets aligned to the ``time`` axis. Plotted as
        points when supplied.
    dates
        Time index aligned to the ``time`` dim. When omitted, the ``time`` coord on
        the posterior is used (and treated as datetime if it parses as such).
    last_price
        Reference last price; drawn as a dashed horizontal line when finite.
    ticker
        Optional label for the title.
    hdi_probs
        Credible-interval masses to shade, widest first.
    figsize, color, observed_color
        Cosmetic controls.

    Returns
    -------
    arviz_plots.PlotCollection
        The composed collection (already drawn); call ``.show()`` to display.
    """
    post = idata.posterior[state_var]
    if "time" not in post.dims:
        raise ValueError(f"{state_var!r} has no 'time' dim; dims={post.dims}.")
    n_time = post.sizes["time"]

    # Resolve the x-axis: prefer explicit `dates`, else the posterior `time` coord.
    # Only treat the coord as datetime when its dtype actually is one -- otherwise
    # pd.to_datetime would silently coerce a plain integer time index into
    # 1970-epoch timestamps. `bool(np.asarray(...).all())` collapses the mask to a
    # scalar (a bare `.all()` on a Series/Index is ambiguous as a truth value).
    if dates is None and "time" in post.coords:
        coord_vals = np.asarray(post["time"].values)
        if np.issubdtype(coord_vals.dtype, np.datetime64):
            dates = pd.DatetimeIndex(coord_vals)
        elif coord_vals.dtype == object:
            parsed = pd.to_datetime(coord_vals, errors="coerce")
            if not bool(np.asarray(pd.isna(parsed)).all()):
                dates = pd.DatetimeIndex(parsed)
    use_dates = (
        dates is not None
        and len(dates) == n_time
        and not bool(np.asarray(pd.isna(dates)).all())
    )
    x = xr.DataArray(
        mdates.date2num(np.asarray(dates)) if use_dates else np.arange(n_time),
        dims="time",
    )

    median = post.median(("chain", "draw"))
    ds = post.to_dataset()

    pc = azp.PlotCollection.grid(
        ds, backend="matplotlib", figure_kwargs={"figsize": figsize}
    )
    target = pc.get_target(state_var, {})  # raw matplotlib Axes

    # Nested HDI bands: widest first with the lightest alpha so inner masses darken.
    band_alphas = (0.16, 0.28, 0.40, 0.50)
    for prob, alpha in zip(sorted(hdi_probs, reverse=True), band_alphas):
        band = post.azstats.hdi(prob=prob)
        azv.fill_between_y(
            median, target, x=x,
            y_bottom=band.sel(ci_bound="lower"),
            y_top=band.sel(ci_bound="upper"),
            facecolor=color, alpha=alpha, edgecolor="none",
        )

    azv.line_xy(median, target, x=x, y=median, color=color, linewidth=2.2, zorder=4)

    if observed is not None:
        obs = xr.DataArray(np.asarray(observed, dtype="float64"), dims="time")
        azv.scatter_xy(
            median, target, x=x, y=obs,
            color=observed_color, s=34, zorder=6,
            edgecolor="#1e1e1e", linewidth=0.6,
        )

    if last_price is not None and np.isfinite(last_price):
        target.axhline(float(last_price), ls="--", color="#bbbbbb", lw=1.2, zorder=2)

    if use_dates:
        target.xaxis_date()
        target.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))

    azv.labelled_x(median, target, text="as-of date" if use_dates else "time step")
    azv.labelled_y(median, target, text="price target")
    title = "Kalman-smoothed price-target path"
    if ticker:
        title += f" — {ticker}"
    target.set_title(title)

    # Hand-built legend (composition primitives don't auto-register labels).
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    handles = [
        Line2D([0], [0], color=color, lw=2.2, label="posterior median state"),
        Patch(facecolor=color, alpha=0.40,
              label=f"{int(max(hdi_probs) * 100)}% / {int(min(hdi_probs) * 100)}% HDI"),
    ]
    if observed is not None:
        handles.append(Line2D([0], [0], marker="o", linestyle="none",
                              markerfacecolor=observed_color, markeredgecolor="#1e1e1e",
                              label="observed price target"))
    if last_price is not None and np.isfinite(last_price):
        handles.append(Line2D([0], [0], ls="--", color="#bbbbbb", label="last price"))
    target.legend(handles=handles, fontsize=8, loc="best", framealpha=0.25)

    return pc


print("Defined plot_price_target_path() — Kalman price-target path via arviz_plots composition.")

## 2. Exploratory Data Analysis (EDA) — `kalman_df`

Source: `pml.mv_pymc_kalman_pt` (one row per ISIN, filtered to `observed_pt IS NOT NULL`).
We resolve column roles from `pml.vw_pymc_feature_catalogue` and fall back to the
known MV schema where the catalogue has no `kalman_pt` rows yet.

In [ ]:
# Map feature_catalogue -> columns actually present in kalman_df
catalogue = feature_catalogue.copy()
catalogue['present'] = catalogue['feature_alias'].isin(kalman_df.columns)

role_summary = (
    catalogue.groupby(['pymc_role', 'feature_role'])['present']
    .agg(n_columns='size', n_present='sum')
    .reset_index()
)
role_summary

In [ ]:
# Resolve column groups by pymc_role, with canonical fallbacks for the Kalman MV.
present = catalogue.loc[catalogue['present']]
PREDICTOR_COLS = present.loc[present['pymc_role'] == 'mutable_predictor', 'feature_alias'].tolist()
COORD_COLS = present.loc[present['pymc_role'] == 'coord', 'feature_alias'].tolist()
RESPONSE_COLS = present.loc[present['pymc_role'].isin(['response', 'observed']), 'feature_alias'].tolist()

# Canonical schema of pml.mv_pymc_kalman_pt (single source of truth in SQL).
# The extended MV emits a per-trail drift feature for every price_* / price_target_*
# family (mean / high / low / median / raw price / analyst-count / dispersion).
KNOWN_FEATURES = ['feat_pt_drift', 'feat_price_drift',
                  'feat_pt_high_drift', 'feat_pt_low_drift', 'feat_pt_median_drift',
                  'feat_coverage_drift', 'feat_pt_noise_drift',
                  'feat_pt_noise_sigma', 'feat_pt_range_norm',
                  'feat_vol_1m', 'feat_vol_3m', 'feat_vol_6m', 'feat_vol_1y',
                  'feat_total_return_ytd']
for col in KNOWN_FEATURES:
    if col in kalman_df.columns and col not in PREDICTOR_COLS:
        PREDICTOR_COLS.append(col)
if 'observed_pt' in kalman_df.columns and 'observed_pt' not in RESPONSE_COLS:
    RESPONSE_COLS.append('observed_pt')

# Hierarchical classification coords (categorical group effects) are distinct
# from the fiscal-calendar DATE anchors. Both carry pymc_role='coord', but the
# date anchors define the single-security *time axis* (used in section 11) and
# must NOT be treated as categorical effects in the cross-sectional model.
CLASSIFICATION_COORDS = [c for c in (
    'isin', 'ticker', 'region', 'country', 'trading_country',
    'exchange', 'unit', 'style_class', 'size_class', 'sector', 'industry'
) if c in kalman_df.columns]
FISCAL_CALENDAR_COLS = [c for c in (
    'income_statement_report_date', 'next_earnings', 'fy_end_date',
    'next_income_statement_report_date', 'next_fy_end_date', 'expected_report_date'
) if c in kalman_df.columns]
DAY_COUNT_COLS = [c for c in (
    'days_to_next_earnings', 'days_since_last_report', 'days_to_next_fy_end',
    'days_to_next_report', 'days_to_expected_report', 'days_to_fy_end'
) if c in kalman_df.columns]
# Keep only non-date coords as categorical-effect candidates; fall back to the
# curated classification list when the catalogue exposes no plain coords.
COORD_COLS = [c for c in COORD_COLS if c not in FISCAL_CALENDAR_COLS] or CLASSIFICATION_COORDS.copy()

print(f'#predictors     : {len(PREDICTOR_COLS)} -> {PREDICTOR_COLS}')
print(f'#response       : {len(RESPONSE_COLS)} -> {RESPONSE_COLS}')
print(f'#coords         : {len(COORD_COLS)} -> {COORD_COLS}')
print(f'#classification : {len(CLASSIFICATION_COORDS)} -> {CLASSIFICATION_COORDS}')
print(f'#fiscal-calendar: {len(FISCAL_CALENDAR_COLS)} -> {FISCAL_CALENDAR_COLS}')
print(f'#day-count      : {len(DAY_COUNT_COLS)} -> {DAY_COUNT_COLS}')

In [ ]:
# 2.1 Shape, dtype, missingness overview.
eda_overview = pd.DataFrame({
    'dtype': kalman_df.dtypes.astype(str),
    'n_missing': kalman_df.isna().sum(),
    'pct_missing': (kalman_df.isna().mean() * 100).round(1),
    'n_unique': kalman_df.nunique(),
})
print(f'kalman_df shape: {kalman_df.shape}')
eda_overview.sort_values('pct_missing', ascending=False).head(30)

In [ ]:
# 2.2 Expected upside by industry - arviz_plots ridge.
# Replaces the previous 2x2 panel histogram. Distributional view of the raw
# implied upside ((observed_pt / last_price) - 1) per industry, drawn as a
# stacked ridge plot so cross-industry shape differences are immediately visible.
import arviz_plots as azp


_d = kalman_df[(kalman_df['observed_pt'] > 0) & (kalman_df['last_price'] > 0)].copy()
_d['upside_pct'] = (_d['observed_pt'] / _d['last_price'] - 1.0) * 100.0
_d['upside_pct'] = _d['upside_pct'].clip(-100, 200)
_d['industry'] = (_d['industry'] if 'industry' in _d.columns
                  else pd.Series('Unknown', index=_d.index))
_d['industry'] = _d['industry'].fillna('Unknown').astype(str)

# Keep industries with enough names to form a meaningful density.
_counts = _d['industry'].value_counts()
_keep = _counts[_counts >= 5].index.tolist()
_d = _d[_d['industry'].isin(_keep)]

# Pack as (industry, sample) into a Dataset so arviz_plots treats `industry`
# as the ridge facet dimension.
_industries = sorted(_d['industry'].unique())
_max_n = int(_d['industry'].value_counts().max())
_arr = np.full((len(_industries), _max_n), np.nan)
for i, ind in enumerate(_industries):
    vals = _d.loc[_d['industry'] == ind, 'upside_pct'].to_numpy()
    _arr[i, :len(vals)] = vals
_ds_ridge = xr.Dataset(
    {'implied_upside_pct': (('industry', 'sample'), _arr)},
    coords={'industry': _industries},
)

azp.plot_ridge(_ds_ridge, var_names=['implied_upside_pct'], sample_dims=['sample'], combined=True)
plt.suptitle('Implied upside (%) by industry - consensus observed_pt vs last_price',
             y=1.02)
plt.tight_layout()
plt.show()

_d['upside_pct'].describe()


In [ ]:
# 2.3 Classification-coord cardinality.
card = {c: kalman_df[c].nunique() for c in CLASSIFICATION_COORDS}
pd.Series(card).sort_values(ascending=False)

## 3. State-Space Feature Mapping (Kalman semantics)

`KalmanFilterPriceTarget` (in `probabilistic_ml_model/pymc_models/KalmanFilterModel.py`)
is a **single-security time-series** `GaussianRandomWalk` filter: a latent log-price
state with process (`sigma_state`) and observation (`sigma_obs`) noise. `pml.mv_pymc_kalman_pt`
is a **cross-sectional** snapshot - the time axis has been collapsed into per-ISIN
`feat_*` drift / noise columns - so here we reuse the same generative ideas as a
one-step Kalman *measurement update* across the whole panel.

The extended MV computes `target_drift()` for **every** `price_* / price_target_*`
trail, so the drift signal is no longer limited to the consensus mean:

| Role                                                     | MV columns                                                                                                                                             | Where it enters the model                                                                 |
|----------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| **Drift / state-transition mean**                        | `feat_pt_drift`, `feat_price_drift`, `feat_pt_high_drift`, `feat_pt_low_drift`, `feat_pt_median_drift`, `feat_coverage_drift`, `feat_total_return_ytd` | regression slopes `beta` on the latent log-uplift                                         |
| **Observation-noise wideners** (non-negative)            | `feat_pt_range_norm`, `feat_pt_noise_sigma`, `feat_pt_noise_drift`, `feat_vol_{1m,3m,6m,1y}`                                                           | scale the measurement noise `sigma_obs`                                                   |
| **Fiscal-calendar time axis** (DATE coords / day-counts) | `income_statement_report_date`, `next_earnings`, `fy_end_date`, ..., `days_to_*`                                                                       | reconstruct the irregular elapsed-time spacing for the **marginalized** GRW in section 11 |

The fiscal-calendar columns are `pymc_role='coord'` but are **not** categorical
group effects - section 11 uses them to scale the random-walk process variance by
real elapsed time.

In [ ]:
# Map mv_pymc_kalman_pt feat_* columns onto Kalman state-space roles.
# Drift / state-transition mean now spans every price_* / price_target_* trail.
DRIFT_FEATURES = [c for c in ('feat_pt_drift', 'feat_price_drift',
                              'feat_pt_high_drift', 'feat_pt_low_drift',
                              'feat_pt_median_drift', 'feat_coverage_drift',
                              'feat_total_return_ytd')
                  if c in kalman_df.columns]

# LEAKAGE GUARDRAIL: feat_implied_upside = (observed_pt - last_price)/last_price is a
# deterministic function of the RESPONSE (observed_pt) and the anchor (last_price);
# log1p(feat_implied_upside) IS the log-uplift the model targets. It must therefore
# NEVER enter the drift-PREDICTOR matrix - that is circular target leakage which
# inflates apparent fit and destroys calibration. It is used only as the model
# OBSERVATION (section 4/5) and for comparative analytics (section 10).
assert 'feat_implied_upside' not in DRIFT_FEATURES, (
    'feat_implied_upside must not be a drift predictor (target leakage).'
)
NOISE_RANGE_COL = 'feat_pt_range_norm' if 'feat_pt_range_norm' in kalman_df.columns else None
NOISE_SIGMA_COL = 'feat_pt_noise_sigma' if 'feat_pt_noise_sigma' in kalman_df.columns else None
VOL_COLS = [c for c in ('feat_vol_1m', 'feat_vol_3m', 'feat_vol_6m', 'feat_vol_1y')
            if c in kalman_df.columns]

mapping_rows: list[tuple[str, str]] = [
    (c, 'drift / state-transition mean (beta)') for c in DRIFT_FEATURES
]
if NOISE_RANGE_COL:
    mapping_rows.append((NOISE_RANGE_COL, 'observation-noise widener (range)'))
if NOISE_SIGMA_COL:
    mapping_rows.append((NOISE_SIGMA_COL, 'observation-noise widener (consensus sigma)'))
mapping_rows += [(c, 'observation-noise widener (volatility)') for c in VOL_COLS]
mapping = pd.DataFrame(mapping_rows, columns=['mv_column', 'state_space_role'])

print(f'Drift features : {DRIFT_FEATURES}')
print(f'Noise drivers  : range={NOISE_RANGE_COL}, sigma={NOISE_SIGMA_COL}, vol={VOL_COLS}')
mapping

## 4. Build PyMC-Aligned Data Containers

In [ ]:
# 4.0 Filter to rows usable for a log-space state-space model: strictly positive
# observed_pt and last_price, and >=1 contributing analyst. Build categorical coords.
model_df = kalman_df.loc[
    (kalman_df['observed_pt'] > 0)
    & (kalman_df['last_price'] > 0)
    & kalman_df['observed_pt'].notna()
    & kalman_df['last_price'].notna()
].copy().reset_index(drop=True)

if 'n_analysts' in model_df.columns:
    model_df['n_analysts'] = model_df['n_analysts'].fillna(1).clip(lower=1)
else:
    model_df['n_analysts'] = 1.0
print(f'Modelling rows (observed_pt>0 & last_price>0): {len(model_df)}')

isin_labels = model_df['isin'].astype(str).values

# Categorical group effects use ONLY the hierarchical classification coords -
# the fiscal-calendar date coords (FISCAL_CALENDAR_COLS) define the section 11
# time axis and must never be expanded into thousands of categorical levels here.
CATEGORICAL_COORDS = [c for c in CLASSIFICATION_COORDS
                      if c in model_df.columns and c not in ('isin', 'ticker')]
coord_uniques, coord_idx = {}, {}
for col in CATEGORICAL_COORDS:
    labels = model_df[col].fillna('Unknown').astype(str).values
    uniques, idx = np.unique(labels, return_inverse=True)
    coord_uniques[col] = uniques
    coord_idx[col] = idx.astype('int64')
print(f'Categorical coords ({len(CATEGORICAL_COORDS)}): {CATEGORICAL_COORDS}')

# Log-space transition anchor + observation. KalmanFilterModel operates in log
# space to keep strictly-positive prices / targets numerically stable.
log_last = np.log(model_df['last_price'].astype('float64').to_numpy())
log_obs = np.log(model_df['observed_pt'].astype('float64').to_numpy())

# SSOT log-uplift TARGET. feat_implied_upside = (observed_pt - last_price)/last_price
# is the SQL-canonical (NULLIF-safe) implied return to the analyst target, so
# log1p(feat_implied_upside) == log_obs - log_last but is computed once, robustly, in
# SQL. The cross-sectional model (section 5) observes THIS uplift directly - which is
# statistically equivalent to observing log_obs (since log_state = log_last + uplift)
# but removes the fixed log_last offset from both sides of the likelihood and avoids
# rebuilding the ratio in Python. Per-row fallback to the price reconstruction keeps
# the notebook runnable before mv_pymc_kalman_pt is refreshed with the new column.
if 'feat_implied_upside' in model_df.columns:
    _iu = model_df['feat_implied_upside'].astype('float64').to_numpy()
    log_uplift_obs = np.where(
        np.isfinite(_iu), np.log1p(np.clip(_iu, -0.999, None)), log_obs - log_last
    )
    _src = 'feat_implied_upside (SSOT)'
else:
    log_uplift_obs = log_obs - log_last
    _src = 'log_obs - log_last (fallback; feat_implied_upside absent)'
print(f'log_last: {log_last.shape}, log_obs: {log_obs.shape}, '
      f'log_uplift_obs: {log_uplift_obs.shape} [{_src}]')

In [ ]:
# 4.1 Standardised drift-feature matrix (state-transition mean inputs).
X_drift_raw = model_df[DRIFT_FEATURES].astype(float)
X_drift_std = (X_drift_raw - X_drift_raw.mean()) / X_drift_raw.std(ddof=0).replace(0, 1.0)
X_drift = X_drift_std.fillna(0.0).to_numpy()
print(f'Drift matrix X_drift: {X_drift.shape}  ({DRIFT_FEATURES})')

# 4.2 Non-negative observation-noise drivers (measurement-variance wideners).
n_obs = len(model_df)

def _nonneg(col):
    if col and col in model_df.columns:
        return model_df[col].astype('float64').fillna(0.0).clip(lower=0.0).to_numpy()
    return np.zeros(n_obs)

range_norm_xs = _nonneg(NOISE_RANGE_COL)

# Consensus stddev -> relative to last_price (NOT observed_pt, to avoid leakage
# into its own measurement-noise term).
if NOISE_SIGMA_COL and NOISE_SIGMA_COL in model_df.columns:
    _sig = model_df[NOISE_SIGMA_COL].astype('float64').fillna(0.0).to_numpy()
    _lp = np.maximum(model_df['last_price'].astype('float64').to_numpy(), 1e-9)
    noise_cv_xs = np.clip(_sig / _lp, 0.0, None)
else:
    noise_cv_xs = np.zeros(n_obs)

vol_xs = (model_df[VOL_COLS].astype('float64').fillna(0.0).clip(lower=0.0).mean(axis=1).to_numpy()
          if VOL_COLS else np.zeros(n_obs))

n_analysts_xs = model_df['n_analysts'].astype('float64').clip(lower=1).to_numpy()
sqrt_n_xs = np.sqrt(n_analysts_xs)
print(f'noise drivers (mean) — range:{range_norm_xs.mean():.3f}  '
      f'cv:{noise_cv_xs.mean():.3f}  vol:{vol_xs.mean():.3f}')

## 5. Cross-Sectional State-Space Model (log-space Kalman update)

Generative form, per ISIN $i$:

$$\eta_i = \mu_0 + X^{\text{drift}}_i\,\beta + \sum_g u^{(g)}_{[i]} \qquad\text{(hierarchical drift mean)}$$
$$\text{uplift}_i = \eta_i + \sigma_{\text{state}}\,z_i \qquad\text{(latent state innovation)}$$
$$\log s_i = \log(\text{last\_price}_i) + \text{uplift}_i \qquad\text{(latent log fair price-target)}$$
$$\log(\text{observed\_pt}_i) \sim \text{StudentT}\!\left(\nu,\ \log s_i,\ \sigma^{\text{obs}}_i\right)$$
$$\sigma^{\text{obs}}_i = \sigma_{\text{base}}\,\frac{1 + \text{range}_i + \text{cv}_i + \tfrac12\text{vol}_i}{\sqrt{n^{\text{analysts}}_i}}$$

The posterior `expected_pt` $= e^{\log s_i}$ is the **Kalman-smoothed** price target — a
shrinkage between the drift-implied prior ($\text{last\_price}\cdot e^{\eta}$) and the
noisy consensus `observed_pt`, with more analysts / tighter dispersion pulling it
toward the raw consensus. `HalfNormal` state / observation noise and the non-centred
innovation mirror `KalmanFilterPriceTarget`.

In [ ]:
# 5.0 Cross-sectional log-space state-space model builder.

_CANDIDATE_GROUPS = ('region', 'exchange', 'unit', 'style_class', 'size_class', 'sector', 'industry')
GROUP_EFFECTS = [c for c in _CANDIDATE_GROUPS if c in coord_idx]


def build_kalman_pt_model(*, robust: bool = True) -> pm.Model:
    """Cross-sectional log-space state-space price-target model.

    One observation per ISIN: ``log(observed_pt)`` is a noisy measurement of a
    latent log fair-value state anchored at ``log(last_price)`` and shifted by a
    hierarchical, drift-feature-driven log-uplift.

    Parameters
    ----------
    robust : bool
        ``True`` -> Student-t measurement likelihood (default; absorbs analyst
        outliers). ``False`` -> Normal-likelihood twin.
    """
    coords = {'isin': isin_labels, 'drift_feature': DRIFT_FEATURES}
    for col in GROUP_EFFECTS:
        coords[col] = coord_uniques[col]

    with pm.Model(coords=coords) as model:
        log_last_d = pm.Data('log_last_price', log_last, dims='isin')
        Xd = pm.Data('drift_features', X_drift, dims=('isin', 'drift_feature'))
        rng_d = pm.Data('feat_pt_range_norm', range_norm_xs, dims='isin')
        cv_d = pm.Data('feat_pt_noise_cv', noise_cv_xs, dims='isin')
        vol_d = pm.Data('feat_vol_mean', vol_xs, dims='isin')
        sqn = pm.Data('sqrt_n_analysts', sqrt_n_xs, dims='isin')
        log_uplift_obs_d = pm.Data('log_uplift_observed', log_uplift_obs, dims='isin')
        idx_data = {col: pm.Data(f'{col}_idx', coord_idx[col], dims='isin')
                    for col in GROUP_EFFECTS}

        # --- State-transition mean: hierarchical drift regression on log-uplift.
        # Data-informed anchor: centre the global log-uplift on the cross-sectional
        # MEDIAN observed uplift (an aggregate over ISINs, NOT the per-ISIN value ->
        # no leakage). This is the same empirical quantity the prior-predictive check
        # (section 6) compares against; anchoring here tightens the funnel-prone
        # sigma_state/sigma_obs variance ridge and speeds convergence vs a 0 prior.
        mu_anchor = float(np.median(log_uplift_obs))
        mu_global = pm.Normal('mu_global', mu_anchor, 0.25)
        beta = pm.Normal('beta', 0.0, 0.25, dims='drift_feature')
        eta = mu_global + pt.dot(Xd, beta)
        for col in GROUP_EFFECTS:
            sigma_g = pm.HalfNormal(f'sigma_{col}', 0.10)
            z_g = pm.Normal(f'z_{col}', 0.0, 1.0, dims=col)
            ge = pm.Deterministic(f'{col}_effect', sigma_g * z_g, dims=col)
            eta = eta + ge[idx_data[col]]

        # --- Latent log state (non-centred GRW-style innovation; HalfNormal sigma).
        sigma_state = pm.HalfNormal('sigma_state', 0.15)
        z_state = pm.Normal('z_state', 0.0, 1.0, dims='isin')
        log_uplift = pm.Deterministic('log_uplift', eta + sigma_state * z_state, dims='isin')
        log_state = pm.Deterministic('log_state', log_last_d + log_uplift, dims='isin')

        # --- Observation noise (Kalman measurement variance), HalfNormal base.
        sigma_obs_base = pm.HalfNormal('sigma_obs_base', 0.10)
        sigma_obs = pm.Deterministic(
            'sigma_obs',
            sigma_obs_base * (1.0 + rng_d + cv_d + 0.5 * vol_d) / sqn,
            dims='isin')

        # --- Measurement likelihood in log space. We observe the log-UPLIFT directly
        #     (SSOT: log1p(feat_implied_upside)), so the likelihood mean is `log_uplift`,
        #     NOT `log_state`. This is equivalent to observing log(observed_pt) with mean
        #     log_state (= log_last + log_uplift) but drops the fixed log_last offset from
        #     both sides -> better-conditioned and no Python ratio reconstruction.
        if robust:
            nu = pm.Gamma('nu', alpha=2.0, beta=0.1)
            pm.StudentT('log_uplift_obs', nu=nu, mu=log_uplift, sigma=sigma_obs,
                        observed=log_uplift_obs_d, dims='isin')
        else:
            pm.Normal('log_uplift_obs', mu=log_uplift, sigma=sigma_obs,
                      observed=log_uplift_obs_d, dims='isin')

        # --- Screening outputs on the price scale.
        pm.Deterministic('expected_pt', pt.exp(log_state), dims='isin')
        pm.Deterministic('expected_upside', pt.exp(log_uplift) - 1.0, dims='isin')
    return model


kalman_pt_model = build_kalman_pt_model(robust=True)
print(f'sec.5 - cross-sectional state-space model on {len(isin_labels)} ISINs; '
      f'{len(DRIFT_FEATURES)} drift features; group effects: {GROUP_EFFECTS}')
pm.model_to_graphviz(kalman_pt_model)

## 6. Prior Predictive Checks

In [ ]:
_prior_var_names = [
    'mu_global', 'beta', 'sigma_state', 'sigma_obs_base', 'nu',
    *[f'sigma_{g}' for g in GROUP_EFFECTS],
    *[f'{g}_effect' for g in GROUP_EFFECTS],
    'log_uplift', 'log_state', 'sigma_obs',
    'expected_pt', 'expected_upside', 'log_uplift_obs',
]
with kalman_pt_model:
    prior_idata = pm.sample_prior_predictive(
        draws=1500, var_names=_prior_var_names,
        random_seed=RANDOM_SEED, return_inferencedata=True,
    )

# Prior implied upside vs the empirical distribution (sanity scale check).
prior_up = prior_idata.prior['expected_upside'].values.reshape(-1)
emp_up = (model_df['observed_pt'] / model_df['last_price'] - 1.0).to_numpy()
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.clip(prior_up, -1, 2), bins=80, density=True, alpha=0.6,
        label='prior expected_upside')
ax.hist(np.clip(emp_up, -1, 2), bins=80, density=True, histtype='step',
        linewidth=1.5, label='empirical observed_pt/last_price - 1')
ax.set_title('Prior predictive implied upside vs empirical')
ax.legend()
plt.tight_layout()
plt.show()
prior_idata

## 7. Posterior Inference (NUTS)

In [ ]:
# Sampler dispatch - nutpie -> numpyro -> pymc fallback. The cross-sectional
# log-space state-space graph is supported by all three NUTS backends; we try
# them in priority order and fall back on the first installed one that succeeds.
sample_kwargs = dict(
    draws=1000, tune=1000, chains=4, cores=2,
    target_accept=0.95, random_seed=RANDOM_SEED,
    progressbar=True, return_inferencedata=True,
    idata_kwargs={"log_likelihood": False},
)

import importlib.util as _ilu

_candidate_samplers = []
if _ilu.find_spec("nutpie") is not None:
    _candidate_samplers.append("nutpie")
if _ilu.find_spec("numpyro") is not None:
    _candidate_samplers.append("numpyro")
_candidate_samplers.append("pymc")  # always available - pure-Python NUTS
print(f"Available NUTS samplers (in priority order): {_candidate_samplers}")

sampling_errors = []
idata = None
for _sampler in _candidate_samplers:
    try:
        with kalman_pt_model:
            idata = pm.sample(nuts_sampler=_sampler, **sample_kwargs)
        print(f"Sampled successfully with nuts_sampler={_sampler!r}.")
        break
    except Exception as e:  # pragma: no cover - environment-dependent fallback
        sampling_errors.append((_sampler, repr(e)))
        print(f"nuts_sampler={_sampler!r} failed: {e!r}")

if idata is None:
    raise RuntimeError(
        "All NUTS samplers failed:\n"
        + "\n".join(f"  - {s}: {err}" for s, err in sampling_errors)
    )

# Merge prior groups into the posterior idata for one-object downstream access.
from probabilistic_ml_model._pymc_arviz_compat import extend_datatree

idata = extend_datatree(idata, prior_idata)
idata

## 8. Posterior Predictive Checks

In [ ]:
with kalman_pt_model:
    pm.sample_posterior_predictive(
        idata, extend_inferencedata=True,
        random_seed=RANDOM_SEED, progressbar=True,
    )

# (a) Distributional overlay — replicated log-uplift draws vs the observed ECDF.
#     A good fit hugs the observed step function.
pc_ppc = azp.plot_ppc_dist(
    idata,
    group="posterior_predictive",
    var_names=["log_uplift_obs"],
    kind="ecdf",
    num_samples=500,
    backend="matplotlib",
)
pc_ppc.show()

# (b) Calibration — posterior-predictive PIT ECDF. If the model is calibrated the
#     PIT values are ~Uniform(0,1), so the curve tracks the diagonal and stays
#     inside the simultaneous confidence envelope. Systematic excursions flag
#     over-/under-dispersion of the measurement-noise model.
try:
    pc_pit = azp.plot_ppc_pit(
        idata,
        var_names=["log_uplift_obs"],
        backend="matplotlib",
    )
    pc_pit.show()
except Exception as e:  # pragma: no cover - diagnostic is best-effort
    print(f"PPC PIT calibration plot skipped: {e!r}")

## 9. MCMC Diagnostics

In [ ]:
# 9.1 R-hat / ESS summary across the cross-sectional parameter set.
posterior = idata.posterior
requested = ['mu_global', 'beta', 'sigma_state', 'sigma_obs_base', 'nu']
for _grp in GROUP_EFFECTS:
    requested.extend([f'sigma_{_grp}', f'{_grp}_effect'])

available, skipped = [], []
for v in requested:
    if v not in posterior.data_vars:
        skipped.append((v, 'not in posterior'))
        continue
    da = posterior[v]
    non_sample_sizes = [da.sizes[d] for d in da.dims if d not in ('chain', 'draw')]
    if any(s == 0 for s in non_sample_sizes):
        skipped.append((v, f'empty dim(s): {dict(da.sizes)}'))
        continue
    available.append(v)

if skipped:
    print('Skipping variables:')
    for name, reason in skipped:
        print(f'  - {name}: {reason}')
if not available:
    raise RuntimeError('No non-empty variables to summarise.')

summary = azs.summary(idata, var_names=available, round_to=4)
summary.sort_values('r_hat', ascending=False).head(50)

In [ ]:
# 9.2 Divergences and aggregated R-hat / ESS.
n_div = int(idata.sample_stats['diverging'].sum())


def _non_empty_vars(ds):
    keep = []
    for name, da in ds.data_vars.items():
        sizes = [da.sizes[d] for d in da.dims if d not in ('chain', 'draw')]
        if all(s > 0 for s in sizes):
            keep.append(name)
    return keep


posterior_tree = idata.posterior
posterior = posterior_tree.dataset if hasattr(posterior_tree, "dataset") else posterior_tree.to_dataset()
keep_vars = _non_empty_vars(posterior)
rhat_ds = azs.rhat(posterior[keep_vars])
ess_ds = azs.ess(posterior[keep_vars], method='bulk')

max_rhat = float(max(float(rhat_ds[v].max()) for v in rhat_ds.data_vars))
min_ess = float(min(float(ess_ds[v].min()) for v in ess_ds.data_vars))

_grp_keys = [f'sigma_{g}' for g in GROUP_EFFECTS if f'sigma_{g}' in rhat_ds.data_vars]
_grp_report = {v: (float(rhat_ds[v].max()), float(ess_ds[v].min())) for v in _grp_keys}

print(f'Divergences: {n_div}')
print(f'Max R-hat:   {max_rhat:.4f}')
print(f'Min ESS:     {min_ess:.1f}')
if _grp_report:
    print('Group-effect scale diagnostics (max R-hat, min ESS):')
    for v, (r, e) in _grp_report.items():
        print(f'  - {v:>20s}: r_hat={r:.3f}, ess_bulk={e:.1f}')

In [ ]:
# 9.3 Trace + marginal densities for the key scalar / vector parameters.
# Uses arviz_plots.plot_trace (rank-normalised marginals + per-chain traces).
#
# arviz_plots.plot_trace (v1.1.0) can crash when a single call mixes variables
# whose non-sample dimensions differ: the per-chain aesthetic gets broadcast
# across the extra dim and reshape fails with e.g.
# `cannot reshape array of size 12 into shape (4,)` (4 chains x 3 vector coords).
# We therefore (a) classify each variable from its *actual* posterior shape
# rather than a hard-coded list, plotting any variable that carries an extra
# (vector) dim on its own, and (b) keep a defensive fallback that re-plots the
# scalars one figure per variable if the combined call still raises.
post_trace = idata.posterior
_requested = ['mu_global', 'beta', 'sigma_state', 'sigma_obs_base', 'nu',
              *(f'sigma_{g}' for g in GROUP_EFFECTS)]
_trace_vars = [v for v in _requested if v in post_trace.data_vars]

def _extra_dims(_v):
    return [d for d in post_trace[_v].dims if d not in ('chain', 'draw')]

scalar_vars = [v for v in _trace_vars if not _extra_dims(v)]
vector_vars = [v for v in _trace_vars if _extra_dims(v)]

def _show_trace(_vars):
    pc = azp.plot_trace(idata, var_names=_vars, backend='matplotlib')
    pc.show()

if scalar_vars:
    try:
        _show_trace(scalar_vars)
    except ValueError as exc:
        print(f'Combined scalar trace failed ({exc}); plotting per variable.')
        for _sv in scalar_vars:
            _show_trace([_sv])
for _vv in vector_vars:
    _show_trace([_vv])
if not scalar_vars and not vector_vars:
    print('No trace-eligible variables available.')

In [ ]:
# 9.4 Forest plot of the hierarchical group-effect scales and drift slopes.
import arviz_plots as azp

_forest_vars = [f'sigma_{g}' for g in GROUP_EFFECTS if f'sigma_{g}' in idata.posterior.data_vars]
_forest_vars += [v for v in ('beta',) if v in idata.posterior.data_vars]

if _forest_vars:
    azp.plot_forest(idata, var_names=_forest_vars, combined=True)
    plt.title('Group-effect scales (sigma_<coord>) and drift slopes (beta)')
    plt.tight_layout()
    plt.show()
else:
    print('No group-effect / beta variables in posterior - skipped.')

## 10. Expected Price Targets — Posterior Summary

`expected_pt` is the posterior-mean Kalman-smoothed price target (price units);
`expected_upside_pct` is the implied move vs `last_price`. The 94% HDI gives the
credible band around each smoothed target.

In [ ]:
# Posterior expected price target (smoothed) + implied upside per ISIN.
post = idata.posterior


def _post_mean(v):
    return post[v].mean(('chain', 'draw')).values


def _post_hdi(v, p=0.94):
    da = post[v].stack(s=('chain', 'draw'))
    lo = da.quantile((1 - p) / 2, dim='s').values
    hi = da.quantile(1 - (1 - p) / 2, dim='s').values
    return lo, hi


exp_pt = _post_mean('expected_pt')
exp_up = _post_mean('expected_upside')
pt_lo, pt_hi = _post_hdi('expected_pt')

results = pd.DataFrame({
    'isin': isin_labels,
    'ticker': model_df.get('ticker'),
    'sector': model_df.get('sector'),
    'last_price': model_df['last_price'].to_numpy(),
    'observed_pt': model_df['observed_pt'].to_numpy(),
    'expected_pt': exp_pt,
    'expected_pt_hdi_lo': pt_lo,
    'expected_pt_hdi_hi': pt_hi,
    'expected_upside_pct': exp_up * 100,
    # Comparative return signals (section 10b): raw analyst-implied upside and
    # realised YTD return, alongside the Kalman-smoothed posterior expected upside.
    'implied_upside_pct': (
        model_df['feat_implied_upside'].to_numpy() * 100
        if 'feat_implied_upside' in model_df.columns
        else (model_df['observed_pt'] / model_df['last_price'] - 1.0).to_numpy() * 100
    ),
    'total_return_ytd_pct': (
        model_df['feat_total_return_ytd'].to_numpy() * 100
        if 'feat_total_return_ytd' in model_df.columns else np.nan
    ),
    'n_analysts': model_df['n_analysts'].to_numpy(),
})
results = results.sort_values('expected_upside_pct', ascending=False).reset_index(drop=True)
print(f'Expected price targets for {len(results)} ISINs.')
results.head(50)

In [ ]:
# Shrinkage view: Kalman-smoothed expected_pt vs raw consensus observed_pt.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(results['observed_pt'], results['expected_pt'], s=8, alpha=0.4)
hi = float(np.nanquantile(results['observed_pt'], 0.99))
ax.plot([0, hi], [0, hi], '--', color='#888888', linewidth=1)
ax.set_xlim(0, hi)
ax.set_ylim(0, hi)
ax.set_xlabel('consensus observed_pt')
ax.set_ylabel('smoothed expected_pt')
ax.set_title('Kalman-smoothed expected target vs raw consensus')
plt.tight_layout()
plt.show()

In [ ]:
# Per-industry expected_upside posterior - arviz_plots forest with HDIs.
# Aggregates the ISIN-level `expected_upside` posterior samples by industry
# (mean across ISINs within each industry, per draw) so the forest plot
# shows the posterior mean and 94% HDI of the *industry-level* expected upside.
import arviz_plots as azp
import xarray as xr

eu = idata.posterior['expected_upside'] * 100.0  # to percent
_industry_per_isin = model_df['industry'].fillna('Unknown').astype(str).to_numpy()
_industry_da = xr.DataArray(
    _industry_per_isin, dims='isin', coords={'isin': eu.coords['isin']},
)
expected_upside_by_industry = (
    eu.groupby(_industry_da.rename('industry')).mean('isin')
)

_ds_forest = xr.Dataset({'expected_upside_pct': expected_upside_by_industry})

azp.plot_forest(_ds_forest, var_names=['expected_upside_pct'], combined=True)
plt.title('Per-industry expected upside (%) - posterior mean and 94% HDI')
plt.tight_layout()
plt.show()


In [ ]:
# Section 10b: comparative stock-returns analytics -
# feat_implied_upside (raw analyst consensus) vs expected_upside (Kalman-smoothed
# posterior) vs feat_total_return_ytd (realised YTD return). Three complementary views.
_comp = results.dropna(subset=['expected_upside_pct']).copy()

# (1) Shrinkage scatter: raw analyst-implied upside (x) vs Kalman-smoothed posterior
#     expected upside (y). Points pulled toward y=0 / off the y=x line are shrunk
#     toward the hierarchical group mean.
fig, ax = plt.subplots(figsize=(6.4, 6.4))
ax.scatter(_comp['implied_upside_pct'], _comp['expected_upside_pct'],
           s=10, alpha=0.45, color='#56b4e9', label='ISIN')
_both = np.r_[_comp['implied_upside_pct'].to_numpy(), _comp['expected_upside_pct'].to_numpy()]
_both = _both[np.isfinite(_both)]
_lo, _hi = float(np.nanpercentile(_both, 1)), float(np.nanpercentile(_both, 99))
ax.plot([_lo, _hi], [_lo, _hi], '--', color='#bbbbbb', lw=1.1, label='y = x (no shrinkage)')
ax.axhline(0, color='#555555', lw=0.8); ax.axvline(0, color='#555555', lw=0.8)
ax.set_xlim(_lo, _hi); ax.set_ylim(_lo, _hi)
ax.set_xlabel('raw implied upside  feat_implied_upside (%)')
ax.set_ylabel('Kalman-smoothed expected upside (%)')
ax.set_title('Posterior shrinkage of analyst-implied upside')
ax.legend(fontsize=8, framealpha=0.25)
plt.tight_layout(); plt.show()

# (2) arviz_plots KDE of the posterior cross-sectional-average expected upside, then
#     an empirical KDE overlay of all three signals for a direct expected-vs-implied-
#     vs-realised comparison.
eu_pct = idata.posterior['expected_upside'] * 100.0          # (chain, draw, isin)
try:
    _dist = xr.Dataset({'expected_upside_pct': eu_pct.mean('isin')})
    pc_d = azp.plot_dist(_dist, kind='kde', var_names=['expected_upside_pct'],
                         sample_dims=['chain', 'draw'], backend='matplotlib')
    pc_d.add_title('Cross-sectional avg expected upside (%) - posterior')
    pc_d.show()
except Exception as _e:
    print(f'plot_dist KDE skipped: {_e!r}')

fig, ax = plt.subplots(figsize=(9, 4.2))
for _col, _lab, _c in [
    ('implied_upside_pct', 'raw implied upside (consensus)', '#ffb000'),
    ('expected_upside_pct', 'Kalman-smoothed expected upside', '#56b4e9'),
    ('total_return_ytd_pct', 'realised total return YTD', '#cc79a7'),
]:
    _v = pd.to_numeric(_comp.get(_col), errors='coerce').to_numpy()
    _v = _v[np.isfinite(_v)]
    if _v.size > 5:
        _v = _v[(_v >= np.nanpercentile(_v, 1)) & (_v <= np.nanpercentile(_v, 99))]
    if _v.size:
        sns.kdeplot(_v, ax=ax, label=_lab, color=_c, fill=True, alpha=0.18, lw=1.8)
ax.axvline(0, color='#bbbbbb', ls='--', lw=1.0)
ax.set_xlabel('return / upside (%)')
ax.set_title('Expected vs implied vs realised returns - distributional comparison')
ax.legend(fontsize=8, framealpha=0.25)
plt.tight_layout(); plt.show()

# (3) Per-sector forest-style comparison: posterior expected upside (mean + 94% HDI)
#     against the per-sector mean raw implied upside and realised YTD return. Built
#     with a matplotlib error-bar so the HDI and the two reference series align
#     robustly on one y-axis (reuses the section-10 groupby-by-coord pattern).
_sector_da = xr.DataArray(
    model_df['sector'].fillna('Unknown').astype(str).to_numpy(),
    dims='isin', coords={'isin': eu_pct.coords['isin']},
)
eu_by_sector = eu_pct.groupby(_sector_da.rename('sector')).mean('isin')
_stack = eu_by_sector.stack(s=('chain', 'draw'))
_sec = [str(s) for s in eu_by_sector['sector'].values]
_mean = _stack.mean('s').values
_q_lo = _stack.quantile(0.03, 's').values
_q_hi = _stack.quantile(0.97, 's').values
_ref = (_comp.assign(sector=_comp['sector'].fillna('Unknown').astype(str))
        .groupby('sector')[['implied_upside_pct', 'total_return_ytd_pct']].mean()
        .reindex(_sec))
_order = np.argsort(_mean)
_y = np.arange(len(_sec))
fig, ax = plt.subplots(figsize=(8.5, max(4.0, 0.42 * len(_sec))))
ax.errorbar(_mean[_order], _y,
            xerr=[(_mean - _q_lo)[_order], (_q_hi - _mean)[_order]],
            fmt='o', color='#56b4e9', ecolor='#56b4e9', elinewidth=1.4, capsize=3,
            label='expected upside (posterior mean, 94% HDI)')
ax.scatter(_ref['implied_upside_pct'].to_numpy()[_order], _y, marker='s',
           color='#ffb000', s=34, zorder=6, label='raw implied upside (mean)')
ax.scatter(_ref['total_return_ytd_pct'].to_numpy()[_order], _y, marker='x',
           color='#cc79a7', s=44, zorder=6, label='realised total return YTD (mean)')
ax.axvline(0, color='#bbbbbb', ls='--', lw=1.0)
ax.set_yticks(_y); ax.set_yticklabels(np.array(_sec)[_order])
ax.set_xlabel('return / upside (%)')
ax.set_title('Per-sector: expected vs implied vs realised returns')
ax.legend(fontsize=8, framealpha=0.25, loc='best')
plt.tight_layout(); plt.show()


## 11. Estimated Price Targets Over Time - Single-ISIN Kalman Filter

Sections 5-10 are the **cross-sectional** panel adaptation: one row per ISIN, no time
axis. This section runs the **literal** single-security `GaussianRandomWalk` filter
from `probabilistic_ml_model/pymc_models/KalmanFilterModel.py` to recover the quantity
this model exists for - a **price target evolving over time** with a credible band.

The time axis is reconstructed from the embedded `*_ago` price-target cohort
(`price_target_1w_ago`, `price_target_high_3m_ago`, `price_target_median_1y_ago`, ...),
unpivoted into a `(isin, asof_date, price_target)` panel via
`build_price_target_history()`. The cohort is short (~ 6-16 points) and **irregularly
spaced** (1w, 1m, 3m, 6m, 1y), which is exactly where an explicit latent random walk
funnels.

We therefore fit the **marginalized** parameterization. This is the corrected
integrated-out GRW: the latent path is collapsed analytically into a single
`MvNormal` likelihood whose covariance carries **both** the random-walk (process)
and observation (measurement) variances,

$$\Sigma_{st} = P_0 + \sigma_{\text{state}}^2\,\min(\tau_s, \tau_t) + \sigma_{\text{obs}}^2\,\delta_{st},$$

with $\tau$ the **real elapsed time** between observations (`_resolve_time_deltas`).
The observed log-targets are used *only* as the data - never as the mean - so
`sigma_state` and `sigma_obs` are genuinely identified (the previous revision pinned
the state to the observations, a no-op). The smoothed path is recovered as the
analytic Kalman-smoother mean and rendered with the custom `plot_price_target_path()`
from section 1.1.

The cell is guarded - it degrades cleanly to an informative message if `DB_URL` is
unset or no ISIN carries >= 2 `*_ago` observations.

In [ ]:
# Single-ISIN time-series Kalman filter on the *_ago price-target history.
import os
from pathlib import Path


def _resolve_db_url(env_file: str = 'environment_variables.txt') -> str:
    """Return DB_URL from the environment, falling back to environment_variables.txt.

    The kernel may have been started without sourcing set_env.ps1, in which case
    os.environ has no 'DB_URL'. We then parse the KEY=VALUE lines of the project's
    environment_variables.txt as a fallback so the section still runs.
    """
    url = os.environ.get('DB_URL')
    if url:
        return url

    here = Path.cwd()
    for base in (here, *here.parents):
        candidate = base / env_file
        if candidate.is_file():
            for raw in candidate.read_text(encoding='utf-8').splitlines():
                line = raw.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, _, value = line.partition('=')
                if key.strip() == 'DB_URL':
                    return value.strip().strip('"').strip("'")
            break
    raise KeyError(
        "DB_URL not set in os.environ and not found in environment_variables.txt. "
        "Run `. .\\set_env.ps1` before launching the kernel, or add a DB_URL line."
    )


try:
    from sqlalchemy import create_engine, text
    from probabilistic_ml_model.pymc_models.KalmanFilterModel import KalmanFilterPriceTarget

    engine = create_engine(_resolve_db_url())
    cohort = model_df['isin'].astype(str).tolist()

    # Discover the *_ago price-target history columns that actually exist in
    # pml.pml_df, then pull only those (+ identifiers) for the modelled cohort.
    _hist_re = (r"^(price_target(_high|_low|_median)?|price)"
                r"_(5d|1w|1m|3m|6m|1y|3y|5y|mtd|qtd|ytd)_ago$")
    with engine.connect() as conn:
        hist_cols = pd.read_sql(
            text("""
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'pml' AND table_name = 'pml_df'
                  AND (column_name ~ :pat
                       OR column_name IN ('isin', 'ticker', 'last_price', 'price_target'))
                ORDER BY column_name
            """),
            conn, params={'pat': _hist_re},
        )['column_name'].tolist()

        col_sql = ', '.join(f'"{c}"' for c in hist_cols)
        snap = pd.read_sql(
            text(f'SELECT {col_sql} FROM pml.pml_df WHERE isin = ANY(:isins)'),
            conn, params={'isins': cohort},
        )

    n_ago = sum(c.endswith('_ago') for c in hist_cols)
    print(f'Pulled pml.pml_df history frame: {snap.shape}  ({n_ago} *_ago columns).')

    # Unpivot *_ago -> long (isin, asof_date, price_target); pick richest ISIN in
    # the cohort. now_cols omits last_price so the target series is not polluted
    # by the spot price (last_price is shown separately as the reference line).
    long_df, eligible, date_col = KalmanFilterPriceTarget.build_price_target_history(
        snap, now_cols=('price_target',),
    )
    chosen = KalmanFilterPriceTarget.select_target_isin(eligible, cohort=cohort)

    if chosen is None or date_col is None:
        print('No ISIN has >= 2 *_ago price-target observations; section skipped.')
    else:
        ts = (long_df.loc[long_df['isin'] == chosen, ['asof_date', 'price_target']]
              .dropna().sort_values('asof_date').reset_index(drop=True))
        dates = pd.DatetimeIndex(ts['asof_date'])
        observed = ts['price_target'].to_numpy()

        _row = model_df.loc[model_df['isin'] == chosen]
        ticker = str(_row['ticker'].iloc[0]) if len(_row) else str(chosen)
        last_price = float(_row['last_price'].iloc[0]) if len(_row) else None

        print(f'Fitting single-ISIN Kalman filter for {chosen} ({ticker}) '
              f'on {len(observed)} observations spanning '
              f'{dates.min():%Y-%m-%d} … {dates.max():%Y-%m-%d}.')

        kf = KalmanFilterPriceTarget()
        # Short, irregular *_ago cohort -> use the corrected *marginalized* GRW
        # (the latent path is integrated out into the MvNormal covariance, scaled
        # by real elapsed time). This is what parameterization='auto' selects for
        # series below the short-series threshold; we set it explicitly so the
        # section deterministically exercises the funnel-free path.
        kf_idata, kf_model = kf.fit(
            price_targets=observed, isin=str(chosen), dates=dates,
            samples=2500, tune=2000, chains=8,
            random_seed=RANDOM_SEED, parameterization='marginalized',
            target_accept=0.95, nuts_sampler='nutpie',
        )

        n_div = int(kf_idata.sample_stats['diverging'].sum())
        print(f'Marginalized GRW fit: {len(observed)} obs, '
              f'{dates.max().year - dates.min().year}y span, divergences={n_div}.')
        display(azs.summary(kf_idata,
                            var_names=['sigma_state', 'sigma_obs', 'log_state_init'],
                            round_to=4))

        # Headline custom plot — Kalman-smoothed price-target path over time.
        pc_path = plot_price_target_path(
            kf_idata, observed=observed, dates=dates,
            last_price=last_price, ticker=ticker,
        )
        pc_path.show()
except Exception as e:  # pragma: no cover - optional / environment-dependent
    print(f'Section 11 (single-ISIN time-series Kalman) skipped: {e!r}')

## 12. Estimated Price Targets over upcoming/recent Earnings Period - Mingle-ISIN Kalman Filter

Section 11 fits the literal single-security `GaussianRandomWalk` filter on the one ISIN
with the richest `*_ago` history. This section keeps the same state-space machinery but
re-frames **what the time axis represents**: instead of one security's history, it builds
a **mingled cross-sectional consensus** over every ISIN whose `next_earnings` lands in the
**recent earnings window**

```sql
SELECT *

WHERE next_earnings >= current_date - INTERVAL '1 week'
  AND next_earnings <= current_date + INTERVAL '1 week'
```

i.e. names reporting in the ±1-week band around today. For each such ISIN we unpivot the
embedded `*_ago` price-target cohort (`price_target_1w_ago`, `price_target_high_3m_ago`,
`price_target_median_1y_ago`, …) into a `(isin, asof_date, price_target)` panel via
`KalmanFilterPriceTarget.build_price_target_history()`, then **mingle the ISINs** by taking
the cross-sectional **median** price target at each shared `asof_date`. The result is a
single, time-ordered series — the earnings-cohort consensus price target as it evolved over
the recent earnings period — which is exactly the quantity the Kalman filter smooths.

The cohort series is short (~6-16 points) and **irregularly spaced** (1w, 1m, 3m, 6m, 1y),
so we fit the **marginalized** parameterization (the integrated-out GRW whose `MvNormal`
covariance carries both the random-walk and observation variances, scaled by real elapsed
time via `_resolve_time_deltas`). This is funnel-free for sparse, irregular cohorts.

**Visual comparison of the three quantities** (`last_price`, `observed_price_target`,
`expected_pt`):

- `expected_pt` — the posterior-mean Kalman-smoothed latent state (`state` in price space),
  with 94 % / 50 % HDI bands.
- `observed_price_target` — the mingled cohort-median consensus target at each `asof_date`.
- `last_price` — the cohort-median spot price, drawn as a dashed reference line.

Rendered as (a) the headline `plot_price_target_path()` time-series composition, (b) an
ArviZ `plot_forest` of the per-`asof_date` `expected_pt` posterior HDIs, and (c) a tidy
comparison table. The cell is guarded — it degrades cleanly to an informative message if
`DB_URL` is unset or the earnings window yields fewer than 2 mingled observations.


In [ ]:
# Mingle-ISIN time-series Kalman filter over the recent-earnings-window cohort.
# Reuses _resolve_db_url(), KalmanFilterPriceTarget, plot_price_target_path() and
# RANDOM_SEED defined earlier in the notebook.
try:
    from sqlalchemy import create_engine, text
    from probabilistic_ml_model.pymc_models.KalmanFilterModel import KalmanFilterPriceTarget

    engine = create_engine(_resolve_db_url())

    # Discover the *_ago price-target history columns present in pml.pml_df, plus
    # the identifiers / reference columns and the next_earnings timing column used
    # to scope the recent-earnings window.
    _hist_re = (r"^(price_target(_high|_low|_median)?|price)"
                r"_(5d|1w|1m|3m|6m|1y|3y|5y|mtd|qtd|ytd)_ago$")
    with engine.connect() as conn:
        hist_cols = pd.read_sql(
            text("""
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'pml' AND table_name = 'pml_df'
                  AND (column_name ~ :pat
                       OR column_name IN ('isin', 'ticker', 'last_price',
                                          'price_target', 'next_earnings'))
                ORDER BY column_name
            """),
            conn, params={'pat': _hist_re},
        )['column_name'].tolist()

        col_sql = ', '.join(f'"{c}"' for c in hist_cols)
        # Time axis = the recent earnings period: names reporting within +/- 5 days
        # of today. next_earnings is the pml.pml_df earnings-timing column.
        snap = pd.read_sql(
            text(f"""
                SELECT {col_sql}
                FROM pml.pml_df
                WHERE next_earnings >= current_date - INTERVAL '5 days'
                  AND next_earnings <= current_date + INTERVAL '5 days'
            """),
            conn,
        )

    n_ago = sum(c.endswith('_ago') for c in hist_cols)
    n_cohort = snap['isin'].nunique() if 'isin' in snap.columns else 0
    print(f'Recent-earnings window cohort: {snap.shape[0]} rows / {n_cohort} ISINs '
          f'({n_ago} *_ago columns).')

    # Unpivot *_ago -> long (isin, asof_date, price_target) for the whole cohort.
    # now_cols omits last_price so the target series is not polluted by spot price.
    long_df, _eligible, date_col = KalmanFilterPriceTarget.build_price_target_history(
        snap, now_cols=('price_target',),
    )

    if date_col is None or long_df.empty:
        print('No *_ago price-target history in the earnings window; section skipped.')
    else:
        # MINGLE the ISINs: cross-sectional median price target at each shared
        # asof_date -> a single earnings-cohort consensus series over time.
        mingled = (
            long_df.groupby('asof_date', as_index=False)
            .agg(price_target=('price_target', 'median'),
                 n_isin=('isin', 'nunique'))
            .sort_values('asof_date')
            .reset_index(drop=True)
        )

        if len(mingled) < 2:
            print(f'Mingled cohort has only {len(mingled)} distinct as-of date(s); '
                  'need >= 2 for a Kalman fit. Section skipped.')
        else:
            dates = pd.DatetimeIndex(mingled['asof_date'])
            observed = mingled['price_target'].to_numpy()
            # Cohort-median spot price as the reference last_price.
            last_price = (float(np.nanmedian(snap['last_price']))
                          if 'last_price' in snap.columns
                          and np.isfinite(np.nanmedian(snap['last_price'])) else None)
            label = f'EARNINGS-COHORT (n={n_cohort})'

            print(f'Fitting mingled-cohort Kalman filter on {len(observed)} '
                  f'consensus observations spanning '
                  f'{dates.min():%Y-%m-%d} ... {dates.max():%Y-%m-%d}.')

            kf = KalmanFilterPriceTarget()
            # Short, irregular cohort -> corrected marginalized (integrated-out) GRW.
            kf_idata, kf_model = kf.fit(
                price_targets=observed, isin=label, dates=dates,
                samples=2500, tune=2000, chains=8,
                random_seed=RANDOM_SEED, parameterization='marginalized',
                target_accept=0.95, nuts_sampler='nutpie',
            )

            n_div = int(kf_idata.sample_stats['diverging'].sum())
            print(f'Marginalized GRW fit: {len(observed)} obs, '
                  f'{(dates.max() - dates.min()).days}d span, divergences={n_div}.')
            display(azs.summary(
                kf_idata,
                var_names=['sigma_state', 'sigma_obs', 'log_state_init'],
                round_to=4))

            # (a) Headline composition: expected_pt smoothed path + HDI bands,
            #     observed mingled consensus targets, and last_price reference.
            pc_path = plot_price_target_path(
                kf_idata, observed=observed, dates=dates,
                last_price=last_price, ticker=label,
            )
            pc_path.show()

            # (b) ArviZ forest of the per-as-of-date expected_pt (latent `state`)
            #     posterior HDIs, with the cohort last_price as a reference line.
            _state = kf_idata.posterior['state']
            _state = _state.assign_coords(
                time=[d.strftime('%Y-%m-%d') for d in dates]
            )
            azp.plot_forest(_state.to_dataset(), var_names=['state'], combined=True)
            if last_price is not None:
                plt.axvline(last_price, ls='--', color='#bbbbbb', lw=1.2,
                            label='cohort last_price')
                plt.legend(fontsize=8, framealpha=0.25)
            plt.title(f'Expected price target (Kalman state) per as-of date - {label}')
            plt.xlabel('expected_pt (price)')
            plt.tight_layout()
            plt.show()

            # (c) Tidy comparison table: last_price vs observed_pt vs expected_pt.
            _post = kf_idata.posterior['state']
            _mean = _post.mean(('chain', 'draw')).values
            _stk = _post.stack(s=('chain', 'draw'))
            _lo = _stk.quantile(0.03, dim='s').values
            _hi = _stk.quantile(0.97, dim='s').values
            comparison = pd.DataFrame({
                'asof_date': dates.strftime('%Y-%m-%d'),
                'n_isin': mingled['n_isin'].to_numpy(),
                'observed_pt': observed,
                'expected_pt': _mean,
                'expected_pt_hdi_lo': _lo,
                'expected_pt_hdi_hi': _hi,
                'last_price': last_price,
            })
            comparison['expected_vs_observed_pct'] = (
                (comparison['expected_pt'] / comparison['observed_pt'] - 1.0) * 100
            )
            print('Mingled-cohort comparison (last_price vs observed_pt vs expected_pt):')
            display(comparison.round(3))
except Exception as e:  # pragma: no cover - optional / environment-dependent
    print(f'Section 12 (mingle-ISIN earnings-window Kalman) skipped: {e!r}')

## 13. Granular Earnings-Cohort Expected-Price Simulation - Posterior-Predictive Forest

Section 12 **mingles** the recent-earnings cohort into a single cross-sectional median series
and refits the literal single-series Kalman filter. This section keeps the cohort definition
identical - **names reporting within +/- 5 days of today** (`next_earnings`) - but goes the
other way: it stays **per-ISIN granular** and re-uses the already-fitted cross-sectional
state-space posterior from sections 5-10 instead of refitting.

For every cohort ISIN the model's log-space measurement likelihood (`log_uplift_obs`, the
posterior-predictive log-uplift) is mapped back to **price units** via
`last_price * exp(log_uplift_obs)`, giving a posterior-predictive distribution of the
*expected stock price* (the simulated analyst price target) for that name. These per-ISIN
predictive distributions are rendered as an `arviz_plots` **posterior-predictive forest**,
with the realised analyst targets overlaid as observation points, following the
`plot_forest(group="posterior_predictive") + visuals.scatter_x` composition pattern.

Two reference layers make the forest readable as a screen:

- **Reference HDI bands** (`add_bands`) - the cohort's central expected-price region, taken
  directly from the **posterior HDIs** of the pooled `expected_pt` latent state (94 % band
  lightest, 50 % band darker). Names whose predictive interval sits entirely outside the
  94 % band are the cohort's relative outliers going into earnings.
- **Cohort `last_price` reference line** (`add_lines`) - the median spot price across the
  cohort, drawn as a dashed vertical line so the simulated targets can be read as implied
  upside / downside.

The cell is guarded - it degrades cleanly to an informative message if `DB_URL` is unset or
no earnings-window ISIN overlaps the fitted cross-sectional posterior. When the cohort is
large the forest is capped to the most extreme names by expected upside (top/bottom 20) to
keep one row per ISIN legible; the truncation is announced.


In [ ]:
# Granular per-ISIN posterior-predictive forest of simulated expected prices for the
# recent-earnings cohort (next_earnings within +/-5 days). Re-uses the fitted
# cross-sectional `idata`, `results`, `model_df`, `_resolve_db_url()` and RANDOM_SEED.
try:
    from sqlalchemy import create_engine, text

    # 1. Recent-earnings cohort: names reporting within +/- 5 days of today.
    engine = create_engine(_resolve_db_url())
    with engine.connect() as conn:
        cohort_meta = pd.read_sql(
            text("""
                SELECT isin, ticker, last_price, next_earnings
                FROM pml.pml_df
                WHERE next_earnings >= current_date - INTERVAL '5 days'
                  AND next_earnings <= current_date + INTERVAL '5 days'
            """),
            conn,
        )
    cohort_isins_all = cohort_meta['isin'].astype(str).unique().tolist()
    print(f'Recent-earnings cohort (next_earnings +/-5d): {len(cohort_isins_all)} ISINs.')

    # 2. Keep only cohort ISINs that are present in the fitted cross-sectional posterior.
    pp = idata.posterior_predictive          # DataTree node: var `log_uplift_obs` over `isin`
    obsd = idata.observed_data
    modelled = set(pp['log_uplift_obs'].coords['isin'].astype(str).values.tolist())
    cohort_isins = [i for i in cohort_isins_all if i in modelled]
    if not cohort_isins:
        raise RuntimeError(
            'No earnings-window ISIN overlaps the fitted cross-sectional posterior '
            f'({len(cohort_isins_all)} cohort ISINs, {len(modelled)} modelled).'
        )
    print(f'Cohort ISINs overlapping the fitted posterior: {len(cohort_isins)}.')

    # Cap the forest so one-row-per-ISIN stays legible; rank by expected upside and
    # keep the most extreme names (top/bottom) when the cohort is large.
    MAX_FOREST = 40
    cohort_results = (results[results['isin'].isin(cohort_isins)]
                      .sort_values('expected_upside_pct', ascending=False))
    if len(cohort_results) > MAX_FOREST:
        half = MAX_FOREST // 2
        keep = pd.concat([cohort_results.head(half), cohort_results.tail(half)])
        print(f'Cohort has {len(cohort_results)} ISINs; showing the {MAX_FOREST} most '
              f'extreme by expected upside (top/bottom {half}).')
    else:
        keep = cohort_results
    forest_isins = keep['isin'].astype(str).tolist()

    # 3. Simulate expected stock prices: exp() the log-space posterior-predictive draws
    #    and the observed analyst targets back into price units, subset to the cohort.
    # `log_uplift_obs` is the posterior-predictive LOG-UPLIFT; map it back to price
    #    units via price = last_price * exp(log_uplift). Align last_price by ISIN.
    _lp = (cohort_meta.assign(isin=cohort_meta['isin'].astype(str))
           .drop_duplicates('isin').set_index('isin')['last_price']
           .reindex(forest_isins).astype('float64'))
    lp_da = xr.DataArray(_lp.to_numpy(), dims='isin', coords={'isin': forest_isins})
    pp_price = (np.exp(pp['log_uplift_obs'].sel(isin=forest_isins)) * lp_da).rename('expected_price')
    obs_price = (np.exp(obsd['log_uplift_obs'].sel(isin=forest_isins)) * lp_da).rename('expected_price')
    ppc_tree = xr.DataTree.from_dict({
        'posterior_predictive': pp_price.to_dataset(),
        'observed_data': obs_price.to_dataset(),
    })

    # 4. Reference bands from the POSTERIOR HDIs: pool the cohort's `expected_pt` latent
    #    state draws and take the 94% / 50% HDI as the central expected-price region.
    exp_pt_pool = (idata.posterior['expected_pt']
                   .sel(isin=forest_isins)
                   .stack(s=('chain', 'draw', 'isin')))
    _q = lambda p: float(exp_pt_pool.quantile(p).values)
    band94 = (_q(0.03), _q(0.97))
    band50 = (_q(0.25), _q(0.75))
    # Cohort last_price reference: median spot price across the shown cohort.
    cohort_last_price = float(np.nanmedian(
        cohort_meta.loc[cohort_meta['isin'].isin(forest_isins), 'last_price']
    ))

    # 5. Posterior-predictive forest + observed analyst targets (scatter_x overlay).
    pc = azp.plot_forest(
        ppc_tree, group='posterior_predictive', combined=True,
        labels=['isin'], backend='matplotlib',
    )
    pc.map(
        azv.scatter_x, 'observations',
        data=ppc_tree.observed_data.ds, coords={'column': 'forest'},
        color='#ffb000',
    )
    pc.map(
        azv.labelled_x, 'xlabel', coords={'column': 'forest'},
        text='expected price (simulated)  -  points = observed analyst target',
        ignore_aes='y',
    )

    # 6. Reference HDI bands (94% lightest, 50% darker) + cohort last_price line.
    pc.coords = {'column': 'forest'}
    pc = azp.add_bands(pc, values=[band94],
                       visuals={'ref_band': {'color': '#56b4e9', 'alpha': 0.12}})
    pc = azp.add_bands(pc, values=[band50],
                       visuals={'ref_band': {'color': '#56b4e9', 'alpha': 0.24}})
    pc = azp.add_lines(pc, values=cohort_last_price,
                       visuals={'ref_line': {'color': '#bbbbbb',
                                             'linestyle': '--', 'linewidth': 1.3}})
    pc.show()

    print(f'Cohort expected_pt 94% HDI band: ({band94[0]:.2f}, {band94[1]:.2f});  '
          f'50% HDI band: ({band50[0]:.2f}, {band50[1]:.2f});  '
          f'cohort last_price ref = {cohort_last_price:.2f}.')

    # 7. Tidy per-ISIN summary for the names shown in the forest.
    _cols = ['isin', 'ticker', 'sector', 'last_price', 'observed_pt',
             'expected_pt', 'expected_pt_hdi_lo', 'expected_pt_hdi_hi',
             'expected_upside_pct']
    display(keep[[c for c in _cols if c in keep.columns]]
            .round(3).reset_index(drop=True))
except Exception as e:  # pragma: no cover - optional / environment-dependent
    print(f'Section 13 (granular earnings-cohort posterior-predictive forest) skipped: {e!r}')

### 13.1 Further views - results-dataframe HDI bands and a cohort distribution KDE

Two complementary `arviz_plots` views building on the posterior-predictive forest above
(both reuse `ppc_tree`, `keep`, `forest_isins` and `cohort_last_price` from the previous
cell):

- **(a) Forest with `results`-keyed reference bands** - the same per-ISIN posterior-predictive
  forest of simulated expected prices, but the reference band is now keyed off the **stored
  `results` dataframe** columns rather than re-pooled posterior draws: the band spans the
  **cohort median** of `expected_pt_hdi_lo` ... `expected_pt_hdi_hi`, with the cohort-median
  `expected_pt` and the cohort `last_price` drawn as reference lines. This reads the cohort's
  consensus-smoothed 94 % credible region straight from the section-10 screening table.
- **(b) Cohort distribution KDE** (`plot_dist`, `kind="kde"`, `sample_dims=["draw"]`) - the
  posterior distribution of the **cohort-average expected upside (%)** for the earnings-window
  names, one KDE per chain (the chain-overlay doubles as a soft convergence check). A dashed
  line at 0 % marks break-even versus `last_price`: mass to the right is net implied upside
  across the cohort going into earnings.


In [ ]:
# 13a. Posterior-predictive forest with reference bands keyed off the stored `results`
# dataframe (cohort median of expected_pt_hdi_lo/hi) instead of re-pooled posterior draws.
try:
    ppc_tree, keep, forest_isins, cohort_last_price  # defined by the Section 13 forest cell
except NameError:
    print('Run the Section 13 posterior-predictive forest cell first '
          '(ppc_tree / keep / cohort_last_price not in scope).')
else:
    # Reference band straight from the section-10 screening table: cohort-median of the
    # per-ISIN stored 94% HDI bounds, plus the cohort-median smoothed expected_pt.
    band_lo = float(np.nanmedian(keep['expected_pt_hdi_lo']))
    band_hi = float(np.nanmedian(keep['expected_pt_hdi_hi']))
    band_med = float(np.nanmedian(keep['expected_pt']))

    pc2 = azp.plot_forest(
        ppc_tree, group='posterior_predictive', combined=True,
        labels=['isin'], backend='matplotlib',
    )
    pc2.map(
        azv.scatter_x, 'observations',
        data=ppc_tree.observed_data.ds, coords={'column': 'forest'},
        color='#ffb000',
    )
    pc2.map(
        azv.labelled_x, 'xlabel', coords={'column': 'forest'},
        text='expected price (simulated)  -  band = cohort-median results HDI '
             '[expected_pt_hdi_lo, expected_pt_hdi_hi]',
        ignore_aes='y',
    )
    pc2.coords = {'column': 'forest'}
    # Band from results df; reference lines = cohort-median expected_pt and last_price.
    pc2 = azp.add_bands(pc2, values=[(band_lo, band_hi)],
                        visuals={'ref_band': {'color': '#9b59b6', 'alpha': 0.15}})
    pc2 = azp.add_lines(pc2, values=band_med,
                        visuals={'ref_line': {'color': '#9b59b6', 'linewidth': 1.4}})
    pc2 = azp.add_lines(pc2, values=cohort_last_price,
                        visuals={'ref_line': {'color': '#bbbbbb',
                                              'linestyle': '--', 'linewidth': 1.3}})
    pc2.show()
    print(f'results-df cohort-median 94% HDI band: ({band_lo:.2f}, {band_hi:.2f});  '
          f'median expected_pt = {band_med:.2f};  cohort last_price = {cohort_last_price:.2f}.')


In [ ]:
# 13b. Cohort distribution KDE (arviz_plots.plot_dist): implied upside vs expected return.
# Overlays three KDEs on one axis for the earnings-window cohort (forest_isins):
#   * posterior E[upside]      - cohort-average `expected_upside` per (chain, draw).
#   * prior E[upside]          - same statistic from the §6 prior predictive (shows the update).
#   * consensus implied upside - `feat_implied_upside` = (observed_pt - last_price)/last_price,
#                                drawn as its cross-name dispersion across the cohort.
# Reference lines: 0% break-even and the consensus cohort-mean implied upside.
# NOTE: prior/posterior KDEs are distributions of the cohort *mean* (uncertainty in the mean);
# the consensus KDE is the spread *across names* - same %-upside units, different statistic.
try:
    idata, prior_idata, model_df, forest_isins  # §5/§6 and the §13 forest cell
except NameError:
    print('Run the Section 6 prior-predictive cell and the Section 13 posterior-predictive '
          'forest cell first (idata / prior_idata / model_df / forest_isins not in scope).')
else:
    from matplotlib.lines import Line2D

    VAR = 'cohort_expected_upside_pct'
    _fi = [str(s) for s in forest_isins]


    def _cohort_mean_pct(da):
        """Cohort-average expected upside (%) over forest_isins, keeping (chain, draw)."""
        return (da.sel(isin=forest_isins).mean('isin') * 100.0).rename(VAR)


    def _consensus_upside_pct():
        """Per-name consensus upside (%): SSOT feat_implied_upside, else observed_pt/last_price-1."""
        m = model_df.set_index(model_df['isin'].astype(str))
        if 'feat_implied_upside' in m.columns:
            s, src = m['feat_implied_upside'].astype('float64'), 'feat_implied_upside (SSOT)'
        else:
            s = m['observed_pt'].astype('float64') / m['last_price'].astype('float64') - 1.0
            src = 'observed_pt/last_price - 1 (fallback)'
        return (s.reindex(_fi).dropna() * 100.0), src


    cohort_upside = _cohort_mean_pct(idata.posterior['expected_upside'])
    prior_cohort = _cohort_mean_pct(prior_idata.prior['expected_upside'])
    _cons, _cons_src = _consensus_upside_pct()
    cons_da = xr.DataArray(_cons.to_numpy(), dims='isin',
                           coords={'isin': _cons.index.to_numpy()}).rename(VAR)

    _C_POST, _C_PRIOR, _C_CONS, _C_REF = '#1f77b4', '#ff7f0e', '#2ca02c', '#bbbbbb'
    # (data, sample_dims, line style, legend label) — single source of truth for plot + legend.
    series = [
        (cohort_upside, ['chain', 'draw'], dict(color=_C_POST, linewidth=2.2),
         'posterior E[upside] (cohort mean)'),
        (prior_cohort, ['chain', 'draw'], dict(color=_C_PRIOR, linewidth=2.0, linestyle='--'),
         'prior E[upside] (cohort mean)'),
    ]
    if len(_cons) >= 2:  # need >=2 names for a cross-name KDE
        series.append((cons_da, ['isin'], dict(color=_C_CONS, linewidth=2.2),
                       'consensus implied upside (across names)'))

    # A larger canvas + constrained_layout that *reserves* room for the external
    # legend. The previous (9, 4.5) figure with an outside legend at
    # bbox_to_anchor=(1.02, 1) left no horizontal space for the axes, so
    # constrained_layout collapsed the axes to zero size ("axes sizes collapsed
    # to zero" warning) and the KDE was unreadable.
    pc3 = None
    for da, sample_dims, style, _ in series:
        pc3 = azp.plot_dist(
            da.to_dataset(), kind='kde', var_names=[VAR], sample_dims=sample_dims,
            backend='matplotlib', plot_collection=pc3,
            visuals={'dist': style},
            **({'figure_kwargs': {'figsize': (13, 5.5), 'layout': 'constrained'}}
               if pc3 is None else {}),
        )

    if pc3 is None:  # series always carries posterior + prior, so unreachable
        raise RuntimeError('No KDE series to plot (expected posterior + prior).')
    ax = pc3.get_target(VAR, {})
    fig = ax.get_figure()
    cons_mean = float(_cons.mean()) if len(_cons) else float('nan')
    ax.axvline(0.0, color=_C_REF, linestyle='--', linewidth=1.3, zorder=1)
    if np.isfinite(cons_mean):
        ax.axvline(cons_mean, color=_C_CONS, linestyle=':', linewidth=1.6, zorder=1)

    # Dynamic x-range: clip to robust percentiles so wide prior / consensus tails
    # don't squash the posterior KDE.
    _all = np.concatenate([cohort_upside.values.ravel(), prior_cohort.values.ravel(),
                           _cons.to_numpy()])
    _lo, _hi = np.nanpercentile(_all, [1, 99])
    _pad = 0.05 * (_hi - _lo)
    ax.set_xlim(_lo - _pad, _hi + _pad)

    # Slightly larger, themed decorations for readability.
    ax.set_xlabel('upside vs last_price (%)', fontsize=11)
    ax.set_ylabel('density', fontsize=11)
    ax.tick_params(axis='both', labelsize=9)
    ax.set_title('Cohort upside (%): consensus implied vs expected '
                 'prior/posterior (earnings window +/-5d)', fontsize=12, pad=10)
    handles = [Line2D([0], [0], label=label, **style) for _, _, style, label in series]
    handles += [
        Line2D([0], [0], color=_C_CONS, lw=1.6, ls=':', label=f'consensus cohort mean ({cons_mean:.1f}%)'),
        Line2D([0], [0], color=_C_REF, lw=1.3, ls='--', label='0% break-even'),
    ]
    # Anchor the legend to the *figure* so constrained_layout reserves space for
    # it instead of shrinking the axes; reserve ~22% of the width on the right.
    fig.legend(handles=handles, fontsize=9, loc='upper left',
               bbox_to_anchor=(0.78, 0.97), borderaxespad=0.0, framealpha=0.9)
    try:
        fig.get_layout_engine().set(rect=(0.0, 0.0, 0.76, 1.0))
    except (AttributeError, TypeError):
        pass
    pc3.show()

    _p_pos = float((cohort_upside > 0).mean().values) * 100.0
    _p_vs_cons = (float((cohort_upside > cons_mean).mean().values) * 100.0
                  if np.isfinite(cons_mean) else float('nan'))
    print(f'Consensus implied upside [{_cons_src}]: cohort mean = {cons_mean:.2f}% '
          f'across {len(_cons)} names.')
    print(f'Expected upside (cohort mean): prior = {float(prior_cohort.mean()):.2f}%, '
          f'posterior = {float(cohort_upside.mean()):.2f}%;  '
          f'P(posterior cohort upside > 0) = {_p_pos:.1f}%;  '
          f'P(posterior cohort upside > consensus mean) = {_p_vs_cons:.1f}%.')

## 14. Comprehensive Summary - Recent Earnings Period vs Historical Data

This closing section consolidates the notebook's outputs into a single decision-oriented read
on the **expected price targets for names reporting within +/- 5 days** (`next_earnings`),
benchmarked against the broader **historical / baseline** data:

- **Cross-sectional benchmark** - the earnings cohort vs the rest of the modelled universe
  (the names *not* reporting this week), on expected upside, share positive, credible-band
  width (uncertainty) and Kalman shrinkage vs raw consensus - read from the section-10
  `results` table.
- **Time-series benchmark** - the mingled cohort's analyst-target trail reconstructed from the
  embedded `*_ago` history (section 12): the oldest historical consensus vs the most recent,
  and the latest Kalman-smoothed target's implied upside vs `last_price`.
- **Distributional view** - an `arviz_plots` KDE overlay of the posterior cohort-average vs
  universe-average expected upside, with a 0 % break-even reference line.

All blocks are guarded: the summary degrades to whatever upstream artifacts (`results`,
`cohort_meta`, `comparison`) are present in the kernel, so it still produces a partial read if
the DB-dependent sections were skipped.


In [ ]:
# Section 14: comprehensive earnings-cohort vs historical-data summary. Reuses `results`
# (section 10), `cohort_meta` (section 13), `comparison` (section 12) and `idata` when present.
def _fmt(x, nd=1, suf=''):
    """Format a possibly-NaN/None scalar for narrative output."""
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return 'n/a'
        return f'{x:.{nd}f}{suf}'
    except Exception:
        return 'n/a'


def _label(row):
    t = row.get('ticker')
    return t if isinstance(t, str) and t.strip() else str(row['isin'])


def _band_width_pct(df):
    denom = df['expected_pt'].replace(0, np.nan)
    return (df['expected_pt_hdi_hi'] - df['expected_pt_hdi_lo']) / denom * 100.0


def _shrink_pct(df):
    denom = df['observed_pt'].replace(0, np.nan)
    return (df['expected_pt'] / denom - 1.0) * 100.0


try:
    results
except NameError:
    print('Section 10 `results` not in scope - run sections 5-10 first.')
else:
    universe = results.copy()

    # ---- A. Cross-sectional: earnings cohort vs the rest of the modelled universe ----
    try:
        _cohort_ids = set(cohort_meta['isin'].astype(str))
    except NameError:
        _cohort_ids = None

    if _cohort_ids:
        _in = universe['isin'].astype(str).isin(_cohort_ids)
        cohort, rest = universe[_in].copy(), universe[~_in].copy()
        groups = [('Earnings cohort (+/-5d)', cohort),
                  ('Historical baseline (not reporting)', rest),
                  ('Full universe', universe)]
    else:
        cohort = rest = None
        groups = [('Full universe', universe)]
        print('Section 13 `cohort_meta` not in scope - showing the universe only. '
              'Run Section 13 to populate the earnings cohort comparison.')

    _rows = []
    for label, df in groups:
        if df is None or len(df) == 0:
            continue
        _rows.append({
            'group': label,
            'n_names': int(len(df)),
            'median_upside_%': df['expected_upside_pct'].median(),
            'mean_upside_%': df['expected_upside_pct'].mean(),
            'positive_upside_%': (df['expected_upside_pct'] > 0).mean() * 100.0,
            'median_band_width_%': _band_width_pct(df).median(),
            'median_shrink_vs_consensus_%': _shrink_pct(df).median(),
            'median_n_analysts': (df['n_analysts'].median()
                                  if 'n_analysts' in df else np.nan),
        })
    summary_tbl = pd.DataFrame(_rows).set_index('group').round(2)
    print('Cross-sectional summary - expected price targets by group:')
    display(summary_tbl)

    # Cohort sector tilt (composition of the names reporting this week).
    if cohort is not None and len(cohort) and 'sector' in cohort.columns:
        sector_mix = (cohort.assign(sector=cohort['sector'].fillna('Unknown'))
                      .groupby('sector')
                      .agg(n=('isin', 'size'),
                           median_upside_pct=('expected_upside_pct', 'median'))
                      .sort_values('n', ascending=False).round(2))
        print('\nEarnings-cohort sector tilt:')
        display(sector_mix.head(10))

    # ---- B. Time-series: recent vs historical mingled cohort price-target trail ----
    try:
        _cmp = comparison
    except NameError:
        _cmp = None
    hist_drift = implied_now = first = last = None
    if _cmp is not None and len(_cmp) >= 2:
        first, last = _cmp.iloc[0], _cmp.iloc[-1]
        hist_drift = ((last['observed_pt'] / first['observed_pt'] - 1.0) * 100.0
                      if first['observed_pt'] else np.nan)
        implied_now = ((last['expected_pt'] / last['last_price'] - 1.0) * 100.0
                       if last['last_price'] else np.nan)

    # ---- C. Headline narrative ----
    print('\n' + '=' * 74)
    print('KEY INSIGHTS - recent earnings period vs historical data')
    print('=' * 74)
    if cohort is not None and len(cohort):
        cu = cohort['expected_upside_pct'].median()
        ru = rest['expected_upside_pct'].median() if rest is not None and len(rest) else np.nan
        print(f'- {len(cohort)} names report within +/-5d. Median expected upside '
              f'{_fmt(cu, 1, "%")} vs {_fmt(ru, 1, "%")} for non-reporting names '
              f'(delta {_fmt(cu - ru, 1, " pp")}).')
        print(f'- {_fmt((cohort["expected_upside_pct"] > 0).mean() * 100, 0, "%")} of the cohort '
              f'has positive expected upside; median credible band width '
              f'{_fmt(_band_width_pct(cohort).median(), 1, "%")} '
              f'(universe {_fmt(_band_width_pct(universe).median(), 1, "%")}).')
        sh = _shrink_pct(cohort).median()
        print(f'- Kalman-smoothed targets sit {_fmt(abs(sh), 1, "%")} '
              f'{"above" if sh >= 0 else "below"} raw consensus (median) - shrinkage toward '
              f'the hierarchical group mean.')
        _top = cohort.sort_values('expected_upside_pct', ascending=False).head(3)
        _bot = cohort.sort_values('expected_upside_pct').head(3)
        _names = lambda d: ', '.join(f'{_label(r)} ({_fmt(r["expected_upside_pct"], 0, "%")})'
                                     for _, r in d.iterrows())
        print(f'- Highest expected upside: {_names(_top)}.')
        print(f'- Lowest / most downside : {_names(_bot)}.')
    else:
        print('- Earnings cohort not available (Section 13 was skipped); '
              'cross-sectional cohort insights omitted.')
    if first is not None and last is not None:
        print(f'- Historical target trail ({first["asof_date"]} -> {last["asof_date"]}): mingled '
              f'cohort consensus {"rose" if (hist_drift or 0) >= 0 else "fell"} '
              f'{_fmt(abs(hist_drift), 1, "%")}; latest Kalman-smoothed target implies '
              f'{_fmt(implied_now, 1, "%")} upside vs cohort last price.')
    else:
        print('- Historical mingled trail not available (Section 12 was skipped); '
              'time-series drift insight omitted.')

    # ---- D. Distributional view: cohort vs universe expected upside (arviz_plots KDE) ----
    try:
        if _cohort_ids:
            _eu = idata.posterior['expected_upside'] * 100.0
            _modelled = set(_eu.coords['isin'].values.astype(str).tolist())
            _cohort_post = [i for i in _cohort_ids if i in _modelled]
            if _cohort_post:
                _cohort_avg = _eu.sel(isin=_cohort_post).mean('isin')
                _univ_avg = _eu.mean('isin')
                _stacked = xr.concat(
                    [_cohort_avg, _univ_avg],
                    dim=pd.Index(['earnings_cohort', 'universe'], name='group'),
                ).rename('avg_expected_upside_pct')
                pc_sum = azp.plot_dist(
                    _stacked.to_dataset(), kind='kde',
                    var_names=['avg_expected_upside_pct'],
                    sample_dims=['chain', 'draw'], backend='matplotlib',
                )
                pc_sum.add_title('Expected upside (%): earnings cohort vs universe '
                                 '(posterior cross-sectional average)')
                pc_sum = azp.add_lines(
                    pc_sum, values=0.0,
                    visuals={'ref_line': {'color': '#bbbbbb',
                                          'linestyle': '--', 'linewidth': 1.3}},
                )
                pc_sum.show()
    except Exception as _e:  # pragma: no cover - plot is best-effort
        print(f'Summary KDE overlay skipped: {_e!r}')